# behaviz — presets & the spec system
Exhaustive tour: built-in presets, custom rcParams, the `~/.behaviz` library (`save`/`load`/`list`/`export`/`import`), and `with_*` chaining.

Presets are `PlotSpec`s. The publication ones (`presentation`/`print`/…) carry a full matplotlib rcParams table, so their rich styling lands on **matplotlib/seaborn**; on bokeh only size/dpi carry.

In [ ]:
import os # sandbox the preset library for this demo
import numpy as np
import behaviz as bv
from bokeh.io import output_notebook, show
output_notebook()

def render(ax):
    if bv.get_renderer().name == 'bokeh':
        show(ax)


In [ ]:
t = np.linspace(0, 1, 80)
s1 = np.sin(t * 6)
s2 = np.cos(t * 6) * 0.6

def show_spec(spec):
    """Draw two lines + markers under one spec, on the active backend."""
    
    with bv.canvas(spec=spec) as ax:
        # bv.plot_line(t, s1, spec=spec)
        bv.plot_line(t, s2)
        bv.plot_scatter(t[::3], s1[::3])
        
        render(ax)
        
    

bv.set_renderer('seaborn')

## Built-in presets
Same data, every preset. `PlotSpec.preset(name)`.

### `default`

In [ ]:
show_spec(bv.load_preset('default').with_title('default'))

### `paper`

In [ ]:
show_spec(bv.load_preset('paper').with_title('paper'))

### `poster`

In [ ]:
show_spec(bv.load_preset('poster').with_title('poster'))

### `notebook`

In [ ]:
show_spec(bv.load_preset('notebook').with_title('notebook'))

### `dark`

In [ ]:
show_spec(bv.load_preset('dark').with_title('dark'))

### `presentation`

In [ ]:
show_spec(bv.load_preset('presentation').with_title('presentation'))

### `presentation_dark`

In [ ]:
show_spec(bv.load_preset('presentation_dark').with_title('presentation_dark'))

### `print`

In [ ]:
show_spec(bv.load_preset('print').with_title('print'))

## Custom preset from your own rcParams
`PlotSpec.preset('custom', style_dict=...)` stores the dict and applies it via `plt.style.use`.

In [ ]:
my_rc = {
    'figure.figsize': (8, 5), 'figure.dpi': 120,
    'lines.linewidth': 3, 'axes.facecolor': '#f3f0e9',
    'axes.grid': True, 'grid.linestyle': ':', 'font.size': 13,
}
show_spec(bv.PlotSpec.preset('custom', style_dict=my_rc).with_title('custom rcParams'))

## The `~/.behaviz` preset library

### `list_presets()` — built-ins are always available

In [ ]:
bv.list_presets()

### `load_preset(name)` — by name (user presets shadow built-ins)

In [ ]:
show_spec(bv.load_preset('poster').with_title('loaded: poster'))

### `save_preset(name, spec)` — persist your own

In [ ]:
mine = bv.PlotSpec.preset('paper').with_title('My House Style').with_size((7, 4))
path = bv.save_preset('house', mine)
print('saved ->', path)
print('now listed:', bv.list_presets()['house'])
show_spec(bv.load_preset('house'))

### `export_preset` / `import_preset` — move a preset between machines

In [ ]:
import tempfile, os
shared = os.path.join(tempfile.mkdtemp(), 'house.json')
bv.export_preset('house', shared)            # write standalone JSON
bv.delete_preset('house')                    # gone from the library
print('after delete:', 'house' in bv.list_presets())
bv.import_preset(shared, name='house_back')   # reinstall under a new name
print('reimported:', bv.list_presets()['house_back'])

## `with_*` chaining — tweak any preset immutably
Each `with_*` returns a new spec; the original is untouched.

In [ ]:
spec = (bv.PlotSpec.preset('notebook')
        .with_title('chained')
        .with_xlim(0, 1)
        .with_scale('y', 'symlog')
        .with_fontsize(14))
show_spec(spec)

## Cross-backend note
Presets are matplotlib rcParams. On bokeh only size/dpi carry; fonts/colors/line widths do not.

In [ ]:
bv.set_renderer('matplotlib')  # reset